# 03 · Gold context vs. retrieved context — telling failures apart

**No real gold context exists for this benchmark yet — this notebook demonstrates the pattern on one small, clearly-labeled *synthetic* example instead.**

The pattern: keep a hand-checked gold set of passages, loaded and scored
independently of whatever the pipeline actually retrieved, then score a run
against that gold set. See `04-benchmarks/clinical-retrieval/gold/README.md`
— it is deliberately empty, because a hand-checked "correct passage" for a
clinical question requires clinical judgment, and inventing one would be
actively harmful in a benchmark whose whole point is catching wrong-but-
confident answers.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `diagnose` | The retrieval-vs-generation attribution logic — the guardrail this notebook exists to prove, before anything else is built on it | `diagnose(0.0, 1.0)` → `"retrieval failed"` |
| `toy_generate` | A stand-in generator that only reports facts explicitly present in the context it's given (no API key needed) | `toy_generate(query, [passage_text])` |
| `fact_coverage` | Scores an answer against a list of ground-truth facts by checking each fact's key figures are present | `fact_coverage(answer, ground_truth_facts)` → `(1.0, [...])` |


## Step 1 — locate the repo root and confirm the environment

This notebook lives two levels below the repo root, so the first thing it does is walk up the directory tree to find `nbio.py` and import it — everything else in this notebook depends on `repo_root` being set correctly.

In [ ]:
# This notebook lives two levels below the repo root (01-modules/01-tools/06-bench/),
# and Jupyter starts a kernel with its working directory set to the
# notebook's own folder -- so nbio.py (at the repo root) is not importable
# yet. Walk up until we find it, same logic nbio.bootstrap() uses
# internally once it CAN be imported.
import sys
from pathlib import Path

def _find_repo_root(start):
    root = start.resolve()
    for _ in range(6):
        if (root / "nbio.py").is_file():
            return root
        root = root.parent
    raise RuntimeError("could not locate nbio.py above the current directory")

_repo_root_for_import = _find_repo_root(Path.cwd())
if str(_repo_root_for_import) not in sys.path:
    sys.path.insert(0, str(_repo_root_for_import))

import nbio
repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 2 — why bother with a second arm at all?

Scored against retrieved context alone, a low faithfulness score is
ambiguous: it could mean retrieval failed (nothing relevant was ever found)
or generation failed (something relevant WAS found and the model still
didn't use it). A reviewer asking "which one broke?" gets a guess.

Scored against **both** arms — retrieved context, and a hand-checked gold
passage — the ambiguity resolves:

| retrieved-context score | gold-context score | diagnosis |
|---|---|---|
| low | high | **retrieval** failed — the right passage exists and the generator can use it when given it |
| low | low | **generation** failed too — even the correct passage didn't produce a good answer |
| high | high | working as intended |
| high | low | retrieved context accidentally supported an answer that gold context contradicts -- worth a second look |

## Step 3 — build the attribution logic first, before anything is built on top of it

This table above is the guardrail: it's what lets a reviewer tell a
retrieval failure from a generation failure instead of guessing. Before any
worked example runs through it, it needs to be shown correct on its own, on
plain numbers, for BOTH failure types the table names.

In [ ]:
def diagnose(retrieved_score, gold_score, low=0.5):
    """The four-row attribution table above, as one function -- called on
    two fact-coverage scores in [0, 1] and a `low` cutoff."""
    retrieved_low = retrieved_score < low
    gold_low = gold_score < low
    if retrieved_low and not gold_low:
        return "retrieval failed -- the right passage exists and the generator can use it when given it"
    if retrieved_low and gold_low:
        return "generation failed too -- even the correct passage didn't produce a good answer"
    if not retrieved_low and not gold_low:
        return "working as intended"
    return "retrieved context accidentally supported an answer that gold context contradicts -- worth a second look"

## Step 4 — prove `diagnose` correctly attributes a retrieval failure

Low retrieved-context score, high gold-context score: the right passage
exists (gold proves it), so the gap is in what was retrieved, not in
generation.

In [ ]:
retrieval_failure_case = diagnose(0.0, 1.0)
print("diagnose(0.0, 1.0) ->", retrieval_failure_case)
assert "retrieval failed" in retrieval_failure_case

## Step 5 — prove `diagnose` correctly attributes a generation failure

Low retrieved-context score AND low gold-context score: even the
hand-checked correct passage didn't produce a good answer, so the failure
is in generation too, not retrieval alone.

In [ ]:
generation_failure_case = diagnose(0.0, 0.0)
print("diagnose(0.0, 0.0) ->", generation_failure_case)
assert "generation failed" in generation_failure_case

## Step 6 — sanity-check the remaining two rows of the table

The two failure cases above are the ones this notebook's worked example
will actually hit, but the table has four rows -- check the other two
(working as intended, and the "worth a second look" case) before moving
on.

In [ ]:
working_as_intended_case = diagnose(1.0, 1.0)
second_look_case = diagnose(1.0, 0.0)
print("diagnose(1.0, 1.0) ->", working_as_intended_case)
print("diagnose(1.0, 0.0) ->", second_look_case)
assert working_as_intended_case == "working as intended"
assert "second look" in second_look_case
print("\nall four rows of the attribution table confirmed correct")

## Step 7 — a synthetic worked example to run through the proven logic

**This question, passage, and both answers are made up for this notebook.**
None of it is a real clinical fact-check — it exists only to demonstrate the
comparison mechanics. Real gold entries follow the format in
`04-benchmarks/clinical-retrieval/gold/README.md`.

In [ ]:
# SYNTHETIC -- constructed for this notebook, not a real clinical claim.
synthetic_question = {
    "id": "SYN01",
    "query": "What thread count is recommended for closing a synthetic-fabric practice wound?",
    "ground_truth_facts": [
        "Use a size 4-0 synthetic monofilament suture for the practice closure.",
        "Space sutures 5mm apart along the practice incision.",
    ],
}

# SYNTHETIC gold passage -- in the shape gold/README.md specifies
# (gold/<question_id>/passages.json), but constructed here inline rather
# than read from disk, since gold/ has no real entries yet.
synthetic_gold_passage = {
    "question_id": "SYN01",
    "passages": [
        {
            "text": (
                "For the practice-fabric closure exercise, use a size 4-0 "
                "synthetic monofilament suture, spacing each stitch 5mm "
                "apart along the incision line."
            ),
            "source": "SYNTHETIC -- made up for this notebook, not a citable source",
            "verified_by": "n/a (synthetic example)",
        }
    ],
}

# SYNTHETIC retrieved context -- deliberately weaker: it's on-topic but
# missing the stitch-spacing fact, standing in for a retrieval system that
# found something related but incomplete.
synthetic_retrieved_context = [
    "Practice sutures on synthetic fabric are typically monofilament, "
    "sized for easy handling by a novice."
]

print("question:", synthetic_question["query"])
print("gold passage:", synthetic_gold_passage["passages"][0]["text"])
print("retrieved context:", synthetic_retrieved_context[0])

## Step 8 — define `toy_generate`, the stand-in generator

A deliberately dumb "generator": it can only report facts explicitly
present in the context it was given. Real notebooks would call an LLM here;
this stand-in exists so the notebook runs with no API key.

In [ ]:
def toy_generate(query, context_passages):
    # A deliberately dumb "generator": it can only report facts explicitly
    # present in the context it was given. Real notebooks would call an LLM
    # here; this stand-in exists so the notebook runs with no API key.
    text = " ".join(context_passages)
    facts = []
    if "4-0" in text:
        facts.append("use a size 4-0 synthetic monofilament suture")
    if "5mm" in text:
        facts.append("space sutures 5mm apart")
    if not facts:
        return "The context does not specify suture size or spacing."
    return "; ".join(facts).capitalize() + "."

## Step 9 — run `toy_generate` on both arms and read the real output

Two arms, same question, same generator, different context. In a real run
you'd call your generation function twice; here — with no real backend
configured — this tiny rule is enough to show the mechanics without needing
an API key.

In [ ]:
answer_from_retrieved = toy_generate(synthetic_question["query"], synthetic_retrieved_context)
answer_from_gold = toy_generate(
    synthetic_question["query"],
    [p["text"] for p in synthetic_gold_passage["passages"]],
)

print("answer from RETRIEVED context:", answer_from_retrieved)
print("answer from GOLD context     :", answer_from_gold)

## Step 10 — define `fact_coverage`

A tiny fact-coverage check stands in for a real judge model here (see
`02-deepeval-metrics.ipynb` for the real DeepEval `ContextualRecallMetric`,
which does this properly with an LLM judge). The point is the shape of the
comparison, not this specific scoring function.

In [ ]:
def fact_coverage(answer, ground_truth_facts):
    covered = [fact for fact in ground_truth_facts if _fact_covered(answer, fact)]
    return len(covered) / len(ground_truth_facts), covered

def _fact_covered(answer, fact):
    # crude but transparent: does the answer contain the fact's key figures?
    key_tokens = [tok for tok in fact.replace(",", "").split() if any(ch.isdigit() for ch in tok) or tok in ("4-0", "5mm")]
    if not key_tokens:
        return fact.lower() in answer.lower()
    return all(tok.lower() in answer.lower() for tok in key_tokens)

## Step 11 — score both arms against the ground-truth facts

In [ ]:
retrieved_score, retrieved_covered = fact_coverage(answer_from_retrieved, synthetic_question["ground_truth_facts"])
gold_score, gold_covered = fact_coverage(answer_from_gold, synthetic_question["ground_truth_facts"])

print(f"retrieved-context arm: {retrieved_score:.0%} fact coverage -- {retrieved_covered}")
print(f"gold-context arm:      {gold_score:.0%} fact coverage -- {gold_covered}")

## Step 12 — run the already-proven `diagnose` function on these real scores

`diagnose` was already shown correct on both failure types (Steps 4-5) and
the remaining two rows (Step 6) using plain numbers. Here it runs on the
real scores this worked example just produced, not on a new untested
codepath.

In [ ]:
diagnosis = diagnose(retrieved_score, gold_score)
print("diagnosis:", diagnosis)

## What this looks like once real gold context exists

Once `gold/<question_id>/passages.json` has real entries (see
`04-benchmarks/clinical-retrieval/gold/README.md` for the format and how to
contribute one), this notebook's `synthetic_gold_passage` becomes:

```python
import json
gold_path = repo_root / "04-benchmarks" / "clinical-retrieval" / "gold" / "B01" / "passages.json"
real_gold = json.loads(gold_path.read_text())["passages"]
```

...and the same `fact_coverage` / `diagnose` comparison runs against a real
hand-checked passage instead of a made-up one. Five questions with gold
context (of the 20) is enough to make this arm meaningful — it does not need
to be complete to be useful.